<a name="projet-7"></a>
# PROJET 7 : Loan Default Prediction #

<a name="contenu"></a>
## Contenu Partie N°2 ##
- Import des données
- Suppression des lignes et colonnes inutiles
  - Retrait des lignes de totalisation
  - Vérification des lignes en doublon
  - Vérification des colonnes entièrement à Null
  - Suppression des colonnes "Hardship", "Settlement" ...
  - Suppression des lignes des "prêts conjoints"
- Création de la variable cible (Loan Condition)
- Transformation de variables
- Visualisations
- Data Cleaning
  - Removing Exclusions
  - Missing Value Imputation
  - Removing Outliers
- Correlation Analysis


In [ ]:
import gc
# import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import plotly.express as px
# import scipy.stats as sps
import seaborn as sns

<a name="import-des-donnees"></a>
## Import des données ##


In [ ]:
df = pd.read_parquet("DATA/accepted_2007_to_2018Q4_filtered.parquet")


In [ ]:
# # Création d'un échantillon

# # Sélection et concaténation des 5 premières, 5 dernières lignes et 200000 lignes tirées aléatoirement
# df_echantillon = pd.concat([df.head(5), df.sample(n=200000, random_state=42), df.tail(5)])

# # Sauvegarde de cet échantillon dans un nouveau fichier CSV
# # L'argument index=False évite de sauvegarder les anciens numéros de ligne comme une nouvelle colonne
# df_echantillon.to_csv("DATA/accepted_2007_to_2018Q4_sample_200k.csv", index=False)

# # Vérification des dimensions
# print(f"L'échantillon a été créé avec succès !")
# print(f"Dimensions du nouvel échantillon : {df_echantillon.shape}")

# df = df_echantillon

<a name="exploration-et-data-cleaning"></a>
## Suppression des lignes et colonnes inutiles ##


In [ ]:
# Aperçu des 5 premières lignes

df.head()

In [ ]:
# Aperçu des 5 dernières lignes

df.tail()

<a name="Retrait des lignes de totalisation"></a>
### Retrait des lignes de totalisation ###


In [ ]:
# On cherche les lignes dont l'id contient "Total" (insensible à la casse)
lignes_total = df[df['id'].str.contains("Total", case=False, na=False)]

print(f"Nombre de lignes de totaux explicites détectées : {len(lignes_total)}")
print("\nContenu de la colonne 'id' pour ces lignes :")
print(lignes_total['id'].unique())

In [ ]:
# On ne garde que les lignes qui ne contiennent PAS "Total" dans l'id
df = df[~df['id'].str.contains("Total", case=False, na=False)]

print(f"Lignes de totaux purgées. Nouvelle taille : {len(df)}")

In [ ]:
# Compte des lignes où loan_amnt est manquant
nb_null_loan = df['loan_amnt'].isnull().sum()

print(f"Nombre de lignes parasites (loan_amnt nul) : {nb_null_loan}")

# Visualiser ces lignes pour confirmer qu'il s'agit bien de totaux
print("\nAperçu des colonnes 'id' de ces lignes :")
print(df[df['loan_amnt'].isnull()]['id'].unique())

In [ ]:
# Suppression des lignes où loan_amnt est NaN
df = df.dropna(subset=['loan_amnt'])

# Vérification de la nouvelle taille du dataset
print(f"Nettoyage terminé. Nouvelles dimensions : {df.shape}")

# Vérification visuelle de la fin du fichier
df.tail()

<a name="Vérification des lignes en doublon"></a>
### Vérification des lignes en doublon ###


In [ ]:
# Compter le nombre total de lignes 100% identiques
nb_doublons = df.duplicated().sum()

print(f"Nombre de lignes en doublon détectées : {nb_doublons}")

# Aperçu de ces doublons :
if nb_doublons > 0:
    print("\nAperçu des lignes dupliquées :")
    display(df[df.duplicated()].head())

<a name="Vérification des colonnes entièrement à Null"></a>
### Vérification des colonnes entièrement à Null ###


In [ ]:
# Liste des colonnes 100% nulles

all_null_cols = df.columns[df.isnull().all()].tolist()

if len(all_null_cols) > 0:
    print(f"Il y a {len(all_null_cols)} colonnes entièrement vides :")
    print(all_null_cols)
else:
    print("Aucune colonne n'est entièrement vide.")

In [ ]:
# Suppression de la colonne member_id

df = df.drop(columns=['member_id'])
print("La colonne 'member_id' a été supprimée.")

<a name="Suppression des colonnes Hardship, Settlement et historique"></a>
### Suppression des colonnes "Hardship", "Settlement" et historique du remboursement ###


In [ ]:
# Calcul du nombre de valeurs nulles par colonne
null_counts = df.isnull().sum()

# Affichage des 20 colonnes avec le plus de valeurs manquantes
print("Synthèse des valeurs manquantes (Top 20) :")
print(null_counts.sort_values(ascending=False).head(20))

**Suppression des colonnes "Hardship" (Plans de difficultés financières)**

Ces colonnes concernent les emprunteurs qui ont rencontré de graves problèmes financiers (perte d'emploi, maladie) et qui ont négocié un plan de sauvetage temporaire ("Hardship plan") avec Lending Club :

 *   hardship_reason : La cause de la difficulté (chômage, etc.).
 *   hardship_type : Le type de plan mis en place.
 *   hardship_status : Le statut du plan (actif, terminé, annulé).
 *   hardship_amount : Le montant de la mensualité pendant le plan.
 *   hardship_start_date / hardship_end_date : Dates de début et de fin du plan.
 *   hardship_length : La durée du plan en mois.
 *   hardship_dpd : Nombre de jours de retard de paiement (Days Past Due) au moment du plan.
 *   hardship_loan_status : Le statut du prêt pendant le plan.
 *   hardship_payoff_balance_amount : Le montant total restant à payer à l'issue du plan.
 *   hardship_last_payment_amount : Le dernier paiement effectué sous ce régime.
 *   payment_plan_start_date : Le jour où le plan a officiellement commencé.
 *   deferral_term : Le nombre de mois pendant lesquels le paiement a été repoussé.
 *   orig_projected_additional_accrued_interest : Les intérêts supplémentaires qui vont s'accumuler à cause de ce plan.

**Objectif de la suppression : éviter le Data Leakage** 

Ces informations n'existent que parce que l'emprunteur a déjà cessé de payer normalement. 
Or, le but du Machine Learning est de prédire le défaut au moment où le prêt est accordé, donc moment où toutes ces colonnes sont inexistantes.

In [ ]:
# Liste exhaustive des colonnes "Hardship"
cols_hardship = [
    'orig_projected_additional_accrued_interest',
    'hardship_reason', 
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount', 
    'payment_plan_start_date',
    'hardship_type', 
    'hardship_status', 
    'hardship_start_date',
    'deferral_term', 
    'hardship_amount', 
    'hardship_dpd',
    'hardship_loan_status', 
    'hardship_length', 
    'hardship_end_date',
    'hardship_flag'  
]

# Suppression des colonnes (errors='ignore' évite un plantage si une colonne est déjà absente)
df = df.drop(columns=cols_hardship, errors='ignore')

print("Les colonnes 'Hardship' ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

**Suppression des colonnes "Settlement" (Règlements de dettes)**

Ces colonnes s'activent lorsqu'un prêt a définitivement fait défaut ("Charged Off") et que le service de recouvrement négocie avec l'emprunteur pour récupérer au moins une partie de l'argent (par exemple, solder la dette pour 40% du montant restant) :

 *   settlement_status : Le statut de l'accord de règlement (ex: complet, rompu).
 *   settlement_date : La date à laquelle le recouvrement a accepté le règlement.
 *   settlement_amount : Le montant final convenu pour clore la dette.
 *   settlement_percentage : Le pourcentage de la dette initiale que représente le règlement.
 *   settlement_term : Le nombre de mois accordés pour payer ce règlement.
 *   debt_settlement_flag_date : La date à laquelle le dossier a été marqué comme "en cours de règlement".


**Objectif de la suppression : éviter le Data Leakage** 

Exactement comme pour le groupe "Hardship", un modèle qui voit un montant dans settlement_amount saura à 100% que le prêt est mauvais, rendant la prédiction inutile et biaisée.


In [ ]:
# Liste exhaustive des colonnes "Settlement"
cols_settlement = [
    'settlement_status',
    'debt_settlement_flag_date',
    'settlement_term',
    'settlement_percentage',
    'settlement_date',
    'settlement_amount',
    'debt_settlement_flag'  
]

# Suppression des colonnes (errors='ignore' évite un plantage si une colonne est déjà absente)
df = df.drop(columns=cols_settlement, errors='ignore')

print("Les colonnes 'Settlement' ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

**Suppression des colonnes liée à l'historique de remboursement du prêt et son état actuel**

 Ces variables décrivent l'historique de remboursement du prêt et son état actuel :

 *   out_prncp / out_prncp_inv : Capital restant dû (principal) sur le montant total financé / sur la part financée par les investisseurs.
 *   total_pymnt / total_pymnt_inv : Total des paiements reçus à ce jour pour le montant total financé / pour la part des investisseurs.
 *   total_rec_prncp : Capital (principal) remboursé par l'emprunteur à ce jour.
 *   total_rec_int : Intérêts remboursés par l'emprunteur à ce jour.
 *   total_rec_late_fee : Frais ou pénalités de retard perçus à ce jour.
 *   recoveries : Montants récupérés après que le prêt a été déclaré en perte (procédure de recouvrement post "charge-off").
 *   collection_recovery_fee : Frais de gestion facturés par les agences de recouvrement pour récupérer les fonds.
 *   last_pymnt_amnt : Montant du tout dernier paiement total reçu.
 *   last_pymnt_d : Date (mois) à laquelle le dernier paiement a été enregistré.
 *   last_credit_pull_d : Date la plus récente à laquelle le prêteur a consulté le dossier de crédit de l'emprunteur.
 *   next_pymnt_d : Date du prochain paiement.


**Objectif de la suppression : éviter le Data Leakage** 

Dans un projet de prédiction de défaut de paiement, l'objectif est de déterminer si un emprunteur va faire défaut au moment où il demande son prêt. Or, ces informations (paiements reçus, reliquat du capital, frais de retard) ne sont connues qu'après que le prêt a été accordé.

In [ ]:
# Liste des colonnes de "Data Leakage" (données futures ou liées au comportement de remboursement)
cols_leakage = [
    'out_prncp', 
    'out_prncp_inv', 
    'total_pymnt', 
    'total_pymnt_inv',
    'total_rec_prncp', 
    'total_rec_int', 
    'total_rec_late_fee',
    'recoveries', 
    'collection_recovery_fee', 
    'last_pymnt_amnt',
    'last_pymnt_d', 
    'last_credit_pull_d',
    'next_pymnt_d'
]

# Suppression des colonnes
df = df.drop(columns=cols_leakage, errors='ignore')

print("Les colonnes de 'Data Leakage' ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

**Suppression de la colonne "issue_d"**

Ce n'est techniquement pas du Data Leakage.
L'objectif est de prédire si un emprunteur va faire défaut avant de lui accorder le prêt.
Or, au moment où le client fait sa demande, la date du jour (qui deviendra le mois et l'année de issue_d) est connue.

Mais si l'on donne l'année "2014" ou "2015" à un algorithme, il risque d'apprendre des règles du type : "les prêts de 2014 ont eu beaucoup de défauts, donc l'année 2014 est un risque".

Le problème ? Quand vous déploierez ce modèle en production en 2025 ou 2026, l'algorithme ne saura pas comment interpréter ces nouvelles années qu'il n'a jamais vues pendant son entraînement. Il va donc perdre en performance.

Conserver issue_d dans les features risque d'apporter plus de problèmes (overfitting temporel) que de solutions. 


In [ ]:
# Suppression des colonnes
cols_leakage = ['issue_d']
df = df.drop(columns=cols_leakage, errors='ignore')

print("Les colonnes de 'Data Leakage' ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

<a name="Suppression des lignes des prêts conjoints"></a>
### Suppression des lignes des "prêts conjoints" ###


In [ ]:
# Calcul du nombre de valeurs nulles par colonne
null_counts = df.isnull().sum()

# Affichage des 20 colonnes avec le plus de valeurs manquantes
print("Synthèse des valeurs manquantes (Top 20) :")
print(null_counts.sort_values(ascending=False).head(20))

**Suppression des lignes avec un "Co-emprunteur" (sec_app_... et ..._joint)**

Des colonnes (comme annual_inc_joint ou sec_app_fico_range_low) correspondent aux informations d'un second demandeur lorsque le prêt est fait à deux (demande conjointe).

Or, les prêts conjoints (à deux emprunteurs) ont une dynamique de risque très différente des prêts individuels (les revenus et les dettes sont cumulés). Mélanger les deux dans un même modèle peut embrouiller l'algorithme. 

Dans la mesure où ces prêts conjoints ne représentent que quelques pourcents du dataset, il semble préférable de les retirer.

In [ ]:
# Voir combien il y a de prêts de chaque type
print("Répartition des types de prêts avant filtrage :")
print(df['application_type'].value_counts())

# Ne conserver que les prêts individuels
df = df[df['application_type'] == 'Individual']

# Supprimer la colonne 'application_type' devenue inutile
df = df.drop(columns=['application_type'])

print(f"\nPrêts conjoints supprimés ! Nouvelles dimensions : {df.shape}")

**Suppression des colonnes ne contenant aucune donnée**


In [ ]:
# Liste des colonnes 100% nulles

all_null_cols = df.columns[df.isnull().all()].tolist()

if len(all_null_cols) > 0:
    print(f"Il y a {len(all_null_cols)} colonnes entièrement vides :")
    print(all_null_cols)
else:
    print("Aucune colonne n'est entièrement vide.")

In [ ]:
# On supprime les colonnes qui sont 100% vides
df = df.dropna(axis=1, how='all')
print(f"Purge terminée ! Nouvelles dimensions du dataset : {df.shape}")

In [ ]:
# Calcul du nombre de valeurs nulles par colonne
null_counts = df.isnull().sum()

# Affichage des 20 colonnes avec le plus de valeurs manquantes
print("Synthèse des valeurs manquantes (Top 20) :")
print(null_counts.sort_values(ascending=False).head(20))

**Colonnes contenant du texte libre ou des données non structurées**

 *   emp_title (Titre du poste) : Le nom du poste ou métier renseigné librement par l'emprunteur lors de sa demande. Étant un champ de texte libre, il contient d'innombrables fautes de frappe, abréviations et variantes pour un même métier. En l'éat, il y a trop de valeurs uniques. Sans un traitement lourd, cette variable va faire "planter" ou sur-apprendre les modèles.

 *   desc (Description du prêt) : Un paragraphe rédigé par l'emprunteur pour expliquer pourquoi il a besoin de ce prêt. C'est du texte pur. Bien qu'il puisse contenir du sens, l'extraire demanderait des techniques de traitement qui sortent du cadre d'une modélisation standard.

 *   title (Titre du prêt) : Un titre court donné par l'emprunteur à son prêt (ex: "Debt Consolidation", "Credit Card Payoff"). Il fait doublon avec la colonne "purpose". Cependant, purpose est une liste déroulante contrôlée (donc propre et standardisée), alors que title est un texte libre soumis aux mêmes problèmes que emp_title.

 *   url (Lien URL) : L'adresse web de la page de la demande de prêt sur le site de Lending Club. Un lien web ne contient absolument aucune information statistique ou mathématique exploitable pour évaluer la solvabilité d'un individu.

 *   zip_code (Code postal) : Les trois premiers chiffres du code postal de l'emprunteur (pour des raisons d'anonymat, LC masque les derniers chiffres, ex: "902xx"). Même tronqués, les codes postaux génèrent beaucoup trop de catégories différentes. C'est trop lourd pour l'encodage. De plus, l'information géographique pertinente est déjà capturée de manière beaucoup plus propre par la colonne addr_state (l'État américain).



In [ ]:
# Liste des colonnes de texte libre et métadonnées non structurées
cols_unstructured = [
    'emp_title', 
    'desc', 
    'title', 
    'url', 
    'zip_code'
]

# Suppression des colonnes (errors='ignore' permet d'éviter un plantage si une colonne est déjà absente)
df = df.drop(columns=cols_unstructured, errors='ignore')

print("Les colonnes de texte libre et métadonnées ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

**Suppression de l'Id**

L'identifiant unique n'a aucun pouvoir de prédiction sur le comportement financier de l'emprunteur. Le conserver lors de l'entraînement obligerait le modèle de Machine Learning à chercher des corrélations mathématiques là où il n'y a que du bruit.

In [ ]:
# Suppression de la colonne 'id'
df = df.drop('id', axis=1)


**Suppression de(s) colonne(s) ayant une valeur unique**


In [ ]:
# La colonne policy_code est un indicateur qui précise si le prêt est conforme aux critères d'acceptation publics de Lending Club.
# Valeur = 1 : Le prêt est "publiquement disponible". Cela correspond aux données de prêts acceptés que l'on trouve généralement dans les fichiers "Accepted".
# Valeur = 2 : Le prêt n'est pas publiquement disponible (souvent lié à des produits spécifiques ou des tests de nouveaux modèles de score de crédit par la plateforme).

df.groupby('policy_code').size() \
    .reset_index(name='count') \
    .sort_values(by='count', ascending=False)

Absence de variance : 100 % des lignes ont la valeur "1".
Comme la valeur est la même pour tout le monde, elle n'apporte aucune information pour différencier un bon d'un mauvais payeur.
Elle doit donc être supprimée pour ne pas alourdir inutilement le traitement.


In [ ]:
# Liste des colonnes constantes ou inutiles identifiées
cols_constant = ['policy_code']

# Suppression des colonnes (errors='ignore' évite l'erreur si la cellule est relancée)
df = df.drop(columns=cols_constant, errors='ignore')

print("La colonne 'policy_code' (sans variance) a été supprimée avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

**Suppression de(s) colonne(s) extrêmement corrélées**


In [ ]:
# Sélection et affichage des 10 premières lignes pour comparaison
cols_to_compare = ['loan_amnt', 'funded_amnt', 'funded_amnt_inv']

print("Aperçu des montants :")
print(df[cols_to_compare].head(10))

# Vérification statistique rapide
print("\nStatistiques descriptives :")
print(df[cols_to_compare].describe())


loan_amnt (Montant demandé) : C'est le montant total que l'emprunteur a demandé initialement lors de sa demande de prêt. C'est la valeur de base avant toute intervention de la plateforme ou des investisseurs.

funded_amnt (Montant financé) : C'est le montant total qui a été engagé pour le prêt.

funded_amnt_inv (Montant financé par les investisseurs) : C'est la part du prêt qui a été effectivement financée par des investisseurs individuels (particuliers) sur la plateforme.

Ces trois variables sont extrêmement corrélées (Multicolinéarité). 

Seule loan_amnt est conservée car elle représente l'intention initiale de l'emprunteur.

In [ ]:
# Liste des colonnes redondantes (fortement corrélées à loan_amnt)
cols_redundant = ['funded_amnt', 'funded_amnt_inv']

# Suppression des colonnes (errors='ignore' évite l'erreur si la cellule est relancée)
df = df.drop(columns=cols_redundant, errors='ignore')

print("Les colonnes 'funded_amnt' et 'funded_amnt_inv' (redondantes) ont été supprimées avec succès.")
print(f"Nouvelles dimensions du dataset : {df.shape}")

<a name="variable-cible"></a>
## Création de la variable cible (Loan Condition) ##


In [ ]:
# Création de la variable cible (Loan Condition)
bad_loan = [
    "Charged Off", 
    "Default", 
    "Does not meet the credit policy. Status:Charged Off", 
    "In Grace Period",
    "Late (16-30 days)", 
    "Late (31-120 days)"
]

df['loan_condition_int'] = df['loan_status'].apply(lambda status: 1 if status in bad_loan else 0).astype(int)
df['loan_condition'] = np.where(df['loan_condition_int'] == 0, 'Good Loan', 'Bad Loan')

# Vérification rapide
df.groupby(['loan_status', 'loan_condition', 'loan_condition_int']).size() \
    .reset_index(name='count') \
    .sort_values(by=['loan_condition', 'count'], ascending=[False, False])

<a name="transformation-variables"></a>
## Transformation de variables ##


In [ ]:
# Conversion de l'ancienneté professionnelle
emp_length_mapping = {
    '10+ years': 10,
    '9 years': 9,
    '8 years': 8,
    '7 years': 7,
    '6 years': 6,
    '5 years': 5,
    '4 years': 4,
    '3 years': 3,
    '2 years': 2,
    '1 year': 1,
    '< 1 year': 0.5,
    'n/a': 0
}

df['emp_length_int'] = df['emp_length'].map(emp_length_mapping)

# Vérification rapide
df.groupby(['emp_length', 'emp_length_int']).size() \
    .reset_index(name='count') \
    .sort_values(by=['emp_length_int', 'count'], ascending=[False, False])

In [ ]:
# Suppression de la colonne originale emp_length
df.drop(columns=['emp_length'], inplace=True)

# Vérification de la présence des colonnes restantes
print(f"La colonne 'emp_length' a été supprimée.")
print(f"Colonnes actuelles : {df.columns.tolist()}")

In [ ]:
# Cartographie des régions
state_to_region = {
    'CA': 'West', 'OR': 'West', 'UT': 'West', 'WA': 'West', 'CO': 'West',
    'NV': 'West', 'AK': 'West', 'MT': 'West', 'HI': 'West', 'WY': 'West', 'ID': 'West',
    'AZ': 'SouthWest', 'TX': 'SouthWest', 'NM': 'SouthWest', 'OK': 'SouthWest',
    'GA': 'SouthEast', 'NC': 'SouthEast', 'VA': 'SouthEast', 'FL': 'SouthEast', 'KY': 'SouthEast',
    'SC': 'SouthEast', 'LA': 'SouthEast', 'AL': 'SouthEast', 'WV': 'SouthEast', 'DC': 'SouthEast',
    'AR': 'SouthEast', 'DE': 'SouthEast', 'MS': 'SouthEast', 'TN': 'SouthEast',
    'IL': 'MidWest', 'MO': 'MidWest', 'MN': 'MidWest', 'OH': 'MidWest', 'WI': 'MidWest',
    'KS': 'MidWest', 'MI': 'MidWest', 'SD': 'MidWest', 'IA': 'MidWest', 'NE': 'MidWest',
    'IN': 'MidWest', 'ND': 'MidWest',
    'CT': 'NorthEast', 'NY': 'NorthEast', 'PA': 'NorthEast', 'NJ': 'NorthEast', 'RI': 'NorthEast',
    'MA': 'NorthEast', 'MD': 'NorthEast', 'VT': 'NorthEast', 'NH': 'NorthEast', 'ME': 'NorthEast'
}

df['region'] = df['addr_state'].map(state_to_region)

# Vérification rapide
df.groupby('region').size() \
    .reset_index(name='count') \
    .sort_values(by='count', ascending=False)

In [ ]:
# Préparation des données
data = {
    'region': ['SouthEast', 'West', 'NorthEast', 'MidWest', 'SouthWest'],
    'count': [16561, 15902, 15582, 12019, 8033]
}
df_regions = pd.DataFrame(data)

# Mapping des régions vers les codes d'États US (ISO-3166)
region_map = {
    'SouthEast': ['AL', 'AR', 'FL', 'GA', 'KY', 'LA', 'MS', 'NC', 'SC', 'TN', 'VA', 'WV'],
    'West': ['AK', 'AZ', 'CA', 'CO', 'HI', 'ID', 'MT', 'NV', 'NM', 'OR', 'UT', 'WA', 'WY'],
    'NorthEast': ['CT', 'DE', 'ME', 'MD', 'MA', 'NH', 'NJ', 'NY', 'PA', 'RI', 'VT'],
    'MidWest': ['IL', 'IN', 'IA', 'KS', 'MI', 'MN', 'MO', 'NE', 'ND', 'OH', 'SD', 'WI'],
    'SouthWest': ['OK', 'TX']
}

# Liste dépliée pour avoir une ligne par État
state_data = []
for region, states in region_map.items():
    val = df_regions[df_regions['region'] == region]['count'].values[0]
    for state in states:
        state_data.append({'state': state, 'region': region, 'count': val})

df_map = pd.DataFrame(state_data)

# Création de la carte Choroplèthe
fig = px.choropleth(df_map, 
                    locations='state', 
                    locationmode="USA-states", 
                    color='count',
                    scope="usa",
                    hover_name='region',
                    color_continuous_scale="Viridis",
                    title='Répartition Géographique des Prêts par Région',
                    labels={'count': 'Nombre de Prêts'})

fig.show()

In [ ]:
# Nettoyage de la mémoire 
del df_regions, df_map
gc.collect()

In [ ]:
# Calcul du taux de défaut par État (en s'assurant que loan_status ou loan_condition a été calculé)
# Si cette cellule est placée avant la création de loan_condition_int, utiliser loan_status
bad_loans = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "In Grace Period", "Late (16-30 days)", "Late (31-120 days)"]
df_temp = df.copy()
df_temp['is_bad'] = df_temp['loan_status'].isin(bad_loans).astype(int)

state_risk = df_temp.groupby('addr_state')['is_bad'].mean().reset_index()
state_risk['is_bad'] = state_risk['is_bad'] * 100 # Conversion en pourcentage
state_risk.columns = ['State', 'Default_Rate']

# Création de la carte
fig = px.choropleth(
    state_risk,
    locations='State',
    locationmode="USA-states",
    color='Default_Rate',
    scope="usa",
    title="Taux de défaut de paiement par État (%)",
    color_continuous_scale="Reds",
    labels={'Default_Rate': 'Taux de défaut (%)'}
)

fig.show()

In [ ]:
# Nettoyage de la mémoire
del df_temp, state_risk
gc.collect()

In [ ]:
# Suppression de la colonne addr_state devenue redondante
df.drop(columns=['addr_state'], inplace=True)

# Confirmation et affichage des colonnes restantes
print(f"La colonne 'addr_state' a été supprimée.")
print(f"Colonnes géographiques conservées : {[col for col in df.columns if 'region' in col]}")

In [ ]:
# Taille maximale de l'échantillon souhaitée
sample_size = 2000000

# On retient la taille maximale ou la totalité du dataset s'il est plus petit
n_samples = min(sample_size, len(df))

# Création de l'échantillon
df_sample = df.sample(n=n_samples, random_state=42)

print(f"Taille du dataset original : {len(df)}")
print(f"Taille de l'échantillon conservé pour l'EDA : {len(df_sample)}")

----

## Visualisations


In [ ]:
data_for_tables = df_sample.copy() 

# --- TABLEAU 1 : Statistiques Descriptives ---
# Sélection des colonnes numériques clés
num_cols = ['loan_amnt', 'int_rate', 'annual_inc', 'dti']
desc_stats = data_for_tables[num_cols].describe().transpose()

# Ajout de la médiane et formatage
desc_stats['median'] = data_for_tables[num_cols].median()
table_1 = desc_stats[['mean', 'median', 'std', 'min', 'max']]
table_1.columns = ['Moyenne', 'Médiane', 'Écart-type', 'Min', 'Max']

print("--- TABLEAU 1 : SYNTHÈSE NUMÉRIQUE ---")
print(table_1.round(2))
print("\n")


In [ ]:
# --- TABLEAU 2 : Analyse de la Variable Cible ---
target_counts = data_for_tables['loan_status'].value_counts()
target_perc = data_for_tables['loan_status'].value_counts(normalize=True) * 100

table_2 = pd.DataFrame({
    'Nombre': target_counts,
    'Proportion (%)': target_perc
})

print("--- TABLEAU 2 : DISTRIBUTION DU STATUT DES PRÊTS ---")
print(table_2.round(2))
print("\n")


In [ ]:
# --- TABLEAU 3 : Risque par Grade ---
# Calcul du taux de défaut (Charged Off) par Grade
def calculate_default_rate(series):
    return (series == 'Charged Off').mean() * 100

risk_by_grade = data_for_tables.groupby('grade')['loan_status'].agg(
    Volume='count',
    Taux_de_defaut=calculate_default_rate
)

print("--- TABLEAU 3 : RISQUE PAR GRADE ---")
print(risk_by_grade.round(2))

In [ ]:
# On réinitialise l'index pour nettoyer les doublons générés lors de la concaténation
df = df.reset_index(drop=True)

# 4. Analyse Montant vs Condition (Numérique vs Catégorique)
print("--- TABLEAU 4 : Montant vs Condition ---")
print(df.groupby('loan_condition')['loan_amnt'].describe()[['mean', '50%', '75%']])


In [ ]:
# 5. Analyse Taux vs Condition (Numérique vs Catégorique)
print("\n--- TABLEAU 5 : Taux d'intérêt vs Condition ---")
print(df.groupby('loan_condition')['int_rate'].mean())


In [ ]:
# 6. Analyse Motif vs Condition (Catégorique vs Catégorique)
print("\n--- TABLEAU 6 : Risque par motif (Top 5) ---")
crosstab = pd.crosstab(df['purpose'], df['loan_condition'], normalize='index') * 100
print(crosstab.sort_values(by='Bad Loan', ascending=False).head(5))

In [ ]:
# 7. Distribution des taux d'intérêt par Grade (Violin Plot)

# Définir l'ordre exact souhaité pour l'axe X
ordre_grades = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

plt.figure(figsize=(10, 6))

# Ajout du paramètre "order"
sns.violinplot(
    x='grade', 
    y='int_rate', 
    data=df, 
    palette="muted", 
    inner="quartile",
    order=ordre_grades
)

plt.title("Violin Plot : Distribution des taux d'intérêt par Grade")
plt.xlabel("Grade")
plt.ylabel("Taux d'intérêt (%)")
plt.show()

In [ ]:
# 8. Évolution de la densité des taux par Grade (Ridgeline Plot)

# Configuration pour l'effet de chevauchement transparent
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0)})

# On définit l'ordre pour que la "montagne" descende de A à G
ordre_grades = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

# Création de la grille
g = sns.FacetGrid(df, row="grade", hue="grade", aspect=5, height=1.5, palette="muted", 
                  row_order=ordre_grades, hue_order=ordre_grades)

# Dessin des courbes
g.map(sns.kdeplot, "int_rate", bw_adjust=.5, clip_on=False, fill=True, alpha=0.8, linewidth=1.5)
g.map(sns.kdeplot, "int_rate", clip_on=False, color="w", lw=2, bw_adjust=.5)
g.map(plt.axhline, y=0, lw=2, clip_on=False)

def label_text(x, color, label):
    ax = plt.gca()
    # On positionne le texte tout à gauche (x=0) et un peu au-dessus de la ligne (y=0.2)
    ax.text(0, 0.2, label, fontweight="bold", color=color,
            ha="left", va="center", transform=ax.transAxes)

# On applique cette fonction à notre grille
g.map(label_text, "int_rate")

# Ajustement du chevauchement
g.fig.subplots_adjust(hspace=-0.3)

# Nettoyage des titres et de l'axe Y
g.set_titles("")
g.set(yticks=[], ylabel="")

# On retire uniquement l'axe de gauche
g.despine(left=True)

# Ajout du titre de l'axe X tout en bas
g.set_axis_labels("Taux d'intérêt (%)", "")

plt.suptitle("Ridgeline Plot : Évolution de la densité des taux par Grade", y=0.98)
plt.show()

# Restauration du thème classique pour la suite de votre notebook
sns.set_theme(style="whitegrid")

In [ ]:
# 9. Montants selon la condition du prêt (Box Plot + Jitter)

plt.figure(figsize=(10, 6))
# On dessine la boîte grise en fond (sans les outliers pour plus de clarté)
sns.boxplot(x='loan_condition', y='loan_amnt', data=df, color="lightgray", showfliers=False)

# On ajoute les points individuels par dessus (le jitter éparpille les points horizontalement)
sns.stripplot(x='loan_condition', y='loan_amnt', data=df, color="orange", alpha=0.5, jitter=True)

plt.title("Box Plot + Jitter : Montants selon la condition du prêt")
plt.xlabel("Condition du prêt")
plt.ylabel("Montant du prêt ($)")
plt.show()

In [ ]:
# 10. Taux d'intérêt par Grade (Boxen Plot)

# Définition de l'ordre logique des grades
ordre_grades = ['A', 'B', 'C', 'D', 'E', 'F', 'G']

plt.figure(figsize=(10, 6))

# Ajout du paramètre order=ordre_grades
sns.boxenplot(
    x='grade', 
    y='int_rate', 
    data=df, 
    palette="viridis",
    order=ordre_grades
)

plt.title("Boxen Plot : Taux d'intérêt par Grade")
plt.xlabel("Grade")
plt.ylabel("Taux d'intérêt (%)")
plt.show()

In [ ]:
# Configuration générale pour l'esthétique
sns.set_theme(style="whitegrid")
ordre_grades = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
palette_cible = {'Good Loan': '#2ecc71', 'Bad Loan': '#e74c3c'} # Vert et Rouge

# ==========================================
# 1. Répartition de la Variable Cible
# ==========================================
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='loan_condition', palette=palette_cible, order=['Good Loan', 'Bad Loan'])

# Ajout des pourcentages sur les barres
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontweight='bold')

plt.title("1. Répartition de la Variable Cible (Déséquilibre des classes)")
plt.xlabel("Condition du prêt")
plt.ylabel("Nombre de prêts")
plt.show()


# ==========================================
# 2. Taux de défaut réel par Grade
# ==========================================
plt.figure(figsize=(10, 5))
# Calcul du pourcentage de défauts pour chaque Grade
taux_defaut = df[df['loan_condition'] == 'Bad Loan'].groupby('grade').size() / df.groupby('grade').size() * 100
taux_defaut = taux_defaut.reset_index(name='taux')

sns.barplot(data=taux_defaut, x='grade', y='taux', palette="Reds", order=ordre_grades)
plt.title("2. Taux de défaut réel par Grade")
plt.xlabel("Grade")
plt.ylabel("Taux de défaut (%)")

# Ajout de la valeur exacte au-dessus des barres
for index, row in taux_defaut.iterrows():
    plt.text(ordre_grades.index(row['grade']), row['taux'] + 0.5, f"{row['taux']:.1f}%", color='black', ha="center")
    
plt.show()


# ==========================================
# 3. Matrice de Corrélation
# ==========================================
plt.figure(figsize=(10, 8))
# Sélection uniquement des variables numériques (int, float)
df_num = df.select_dtypes(include=[np.number])

corr = df_num.corr()
# Création d'un masque pour cacher la moitié supérieure (redondante) du tableau
masque = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, mask=masque, annot=True, fmt=".2f", cmap='coolwarm', 
            vmin=-1, vmax=1, square=True, linewidths=.5)
plt.title("3. Matrice de Corrélation des variables numériques")
plt.show()


# ==========================================
# 4. Impact du DTI (Taux d'Endettement)
# ==========================================
plt.figure(figsize=(10, 5))
# Courbe de densité avec un remplissage (fill=True)
sns.kdeplot(data=df, x='dti', hue='loan_condition', common_norm=False, 
            fill=True, palette=palette_cible, alpha=0.5)

plt.title("4. Distribution du Taux d'Endettement (DTI) par condition de prêt")
plt.xlabel("DTI (Debt-to-Income Ratio)")
plt.ylabel("Densité")

# On limite généralement l'axe X à 50 ou 60 car le DTI a souvent des valeurs aberrantes très lointaines
plt.xlim(0, 50) 
plt.show()

In [ ]:
# Purge des DataFrames statistiques intermédiaires et de l'échantillon
del data_for_tables, desc_stats, table_1, table_2, target_counts, target_perc, risk_by_grade
gc.collect()

<a name="Data-Cleaning"></a>
## Data Cleaning

This part includes:


*   Removing Exclusions
*   Missing Value Imputation
*   Removing Outliers

<a name="Removing-exclusions"></a>
### Removing Exclusions


**Delete variables with more than 80% missing values**

There are a lot of columns which have huge chunk of data missing. These columns are not necessary for our analysis. The following part will drop any columns where 20% or more data is missing, which means only columns whose number of non-null values is at least 80% of the total number of rows in the dataset will be retained.

In [ ]:
drop_df = df_sample


In [ ]:
def get_missing_value_stats(input_df):
    df_null = pd.DataFrame({
        'Missing Count': input_df.isnull().sum(),
        'Missing Percent': 100 * input_df.isnull().sum() / len(input_df),
        'Type': input_df.dtypes
    })
    missing_values = df_null[df_null['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False) #改
    return missing_values

def get_value_stats(input_df):
    df_null = pd.DataFrame({
        '#Count': input_df.notna().sum(),
        '%Populated': 100 * input_df.notna().sum() / len(input_df),
        '#Unique Values':input_df.nunique(),
        'Most Common Value': input_df.mode().iloc[0],
        'Type': input_df.dtypes
    })

    missing_values = df_null[df_null['#Count'] > 0].sort_values(by='#Count', ascending=False)

    return missing_values

# Finding the the count and percentage of values that are missing.
get_missing_value_stats(drop_df)

In [ ]:
#drop any columns where over a certain percentage is missing
drop_df = drop_df.dropna(axis=1, thresh=int(0.20*len(drop_df)))
get_missing_value_stats(drop_df)

<a name="Missing-Value-Imputation"></a>
### Missing Value Imputation


<table>
  <tr><th>index</th><th>Count</th><th>Percent</th><th>Type</th><th>Imputation method</th></tr>
  <tr><td>tot_coll_amt</td><td>66689</td><td>24.06</td><td>float64</td><td>0</td></tr>
  <tr><td>total_rev_hi_lim</td><td>66689</td><td>24.06</td><td>float64</td><td>0</td></tr>
  <tr><td>tot_cur_bal</td><td>66689</td><td>24.06</td><td>float64</td><td>0</td></tr>
  <tr><td>emp_length_int</td><td>11101</td><td>4.005</td><td>float64</td><td>median</td></tr>
  <tr><td>last_pymnt_d</td><td>921</td><td>0.332</td><td>object</td><td>mode</td></tr>
  <tr><td>revol_util</td><td>253</td><td>0.091</td><td>float64</td><td></td></tr>
  <tr><td>collections_12_mths_ex_med</td><td>145</td><td>0.052</td><td>float64</td><td></td></tr>
  <tr><td>pub_rec</td><td>29</td><td>0.0104</td><td>float64</td><td>median</td></tr>
  <tr><td>delinq_2yrs</td><td>29</td><td>0.0104</td><td>float64</td><td>mean</td></tr>
  <tr><td>last_credit_pull_d</td><td>24</td><td>0.0086</td><td>object</td><td>mode</td></tr>
  <tr><td>annual_income</td><td>4</td><td>0.00144</td><td>float64</td><td>mean</td></tr>
  <tr><td>income_category</td><td>4</td><td>0.00144</td><td>object</td><td>mode</td></tr>
</table>

In [ ]:
fillna_df = drop_df.copy()

# # for object variables - Get the mode of next payment date and last payment date and the last date credit amount was pulled
# for column in ["last_pymnt_d", "last_credit_pull_d"]:  #, 'income_category'
#     fillna_df[column] = fillna_df.groupby("region")[column].transform(lambda x: x.fillna(x.mode()))

# for numerical variables
# Get the mode on the number of accounts in which the client is delinquent
fillna_df["pub_rec"] = fillna_df.groupby("region")["pub_rec"].transform(lambda x: x.fillna(x.median()))
# Get the mode of the total number of credit lines the borrower has
fillna_df["total_acc"] = fillna_df.groupby("region")["total_acc"].transform(lambda x: x.fillna(x.median()))

fillna_df["emp_length_int"] = fillna_df.groupby("region")["emp_length_int"].transform(lambda x: x.fillna(x.median()))

# Get the mean of the annual income depending on the region the client is located.
fillna_df["annual_inc"] = fillna_df.groupby("region")["annual_inc"].transform(lambda x: x.fillna(x.mean()))
# Mode of credit delinquencies in the past two years.
fillna_df["delinq_2yrs"] = fillna_df.groupby("region")["delinq_2yrs"].transform(lambda x: x.fillna(x.mean()))

In [ ]:
# # for other, fill in with zero
# fillna_df.fillna(0, inplace=True)
# fillna_df.isnull().sum().max() # Maximum number of nulls.

In [ ]:
# Identifier et remplir les colonnes numériques avec le chiffre 0
cols_num = fillna_df.select_dtypes(include=['number']).columns
fillna_df[cols_num] = fillna_df[cols_num].fillna(0)

# Identifier et remplir les colonnes de texte avec le texte "0" (ou "Inconnu", "Missing", etc.)
cols_str = fillna_df.select_dtypes(include=['object', 'str']).columns
fillna_df[cols_str] = fillna_df[cols_str].fillna("0")

# Vérifier le nombre maximum de valeurs nulles restantes
fillna_df.isnull().sum().max()

In [ ]:
len(fillna_df['loan_condition_int'])
# Loan Ratios (Imbalanced classes)
fillna_df['loan_condition_int'].value_counts()/len(fillna_df['loan_condition_int']) * 100

<a name="Removing-Outliers"></a>
### Removing Outliers

Custom thresholds were used to remove outliers
(3-sigma method did not work well)

In [ ]:
# Définition des conditions de filtrage 
mask = (
    (fillna_df['annual_inc'] <= 250000) &
    (fillna_df['dti'] <= 50) &
    (fillna_df['open_acc'] <= 40) &
    (fillna_df['total_acc'] <= 80) &
    (fillna_df['revol_util'] <= 120) &
    (fillna_df['revol_bal'] <= 250000)
)

print("Dataset before removing outlier:", fillna_df.shape)

# Application du masque 
RemoveOutlier_df = fillna_df[mask].reset_index(drop=True)

print("Dataset after removing outlier:", RemoveOutlier_df.shape)

RemoveOutlier_df.head().transpose()

<a name="Correlation-Analysis"></a>
## Correlation Analysis

Correlation analysis was performed on the variables to assess their importance and relationship to the target variable y. This provided insights into the most relevant variables for predicting good vs bad loans.

For the correlation analysis, categorical variables were label encoded to enable numeric correlation values to be calculated. While this encoding can introduce artificial numerical relationships, it provided a convenient quick view of variable importance.

For the actual model building later on, more appropriate encodings like target encoding were used for the categorical variables.

In [ ]:
target_col = target_variable = "loan_condition_int"

In [ ]:
corr_df = RemoveOutlier_df.copy() 

# correlation with y
correlation_with_loan_condition = corr_df.select_dtypes(include=['int64', 'float64']).corr()[target_variable]
sorted_correlation = correlation_with_loan_condition.drop(target_variable).sort_values(ascending=False)

# plot
plt.figure(figsize=(12, 6))
sns.barplot(x=sorted_correlation.index, y=sorted_correlation.values, orient='v')
plt.xlabel('Correlation with{}'.format(target_variable))
plt.title('Features Correlation with{}'.format(target_variable))
plt.xticks(rotation=90)
plt.show()
print(sorted_correlation)

In [ ]:
# Select the variables with the highest correlation with the dependent variable and explore the correlation between them
top_variables = sorted_correlation.abs().nlargest(10).index.tolist()

plt.figure(figsize=(20, 20))
correlation_matrix = RemoveOutlier_df[top_variables].corr()
mask = np.tril(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, annot=True, cmap='bwr', vmin=-1, vmax=1, square=True, linewidths=0.5, mask=mask)
plt.title('Correlation Heatmap for Numeric Columns of Interest')
plt.show()

In [ ]:
# # Further, explore the specific distribution of the relationship between variables under the action of the dependent variable loan_condition_int
# # This runs a bit slowly, so run with caution
sample_corr_df = corr_df[top_variables + [target_variable]].sample(n=1000, random_state=42)
sns.pairplot(sample_corr_df,hue=target_variable, diag_kind='kde',corner=True)

In [ ]:
# Sauvegardes en vue de la partie suivante

# Données

RemoveOutlier_df.to_parquet("DATA/cleaned_data_for_modeling.parquet")

print("Données nettoyées sauvegardées pour la phase de feature engineering.")

# Cible

with open("CONFIG/target_config.txt", "w") as f:
    f.write(target_col)

print(f"Configuration sauvegardée : la cible est '{target_col}'")

In [ ]:
# 1. Liste des DF à supprimer
df_a_purger = [
    'df', 'df_bases', 'df_sample', 'EDA_df', 'data_for_tables', 
    'drop_df', 'fillna_df', 'RemoveOutlier_df', 'corr_df', 'sample_corr_df'
]

# 2. Suppression de l'environnement 
for var in df_a_purger:
    if var in locals():
        del locals()[var]

# 3. Libération de la mémoire
gc.collect()

print("🧹 Mémoire purgée avec succès ! Le notebook est prêt à être fermé.")

----
